# 🔬 LoRA 变体 — QLoRA、DoRA、AdaLoRA 深度对比

**本文目标**：系统对比 LoRA 的主要变体，理解各自的适用场景。

读完这篇你会理解：
- QLoRA: FP4 量化 + LoRA, 24GB 显存微调 65B 模型
- DoRA: 把权重分解为"方向"和"幅度", 学习率解耦
- AdaLoRA: 自动分配每层的 rank
- LoRA+/rsLoRA: 修正初始化中的梯度不对称

## 1. QLoRA — 量化 + LoRA (Dettmers et al., NeurIPS 2023)

### 1.1 核心思想

```
问题: LoRA 仍然需要把预训练权重加载到显存
  LLaMA-65B FP16: 130 GB → 需要 4×A100

QLoRA: 把预训练权重量化到 4-bit, LoRA 权重保持 FP16

显存对比:
  LLaMA-65B 全量微调:     780 GB  (16×A100)
  LLaMA-65B LoRA:         130 GB  (4×A100)
  LLaMA-65B QLoRA(NF4):   48 GB   (1×A100 或 2×RTX 4090) ← !
```

### 1.2 四项关键创新

```
1. NF4 (NormalFloat4):
   一种针对"正态分布权重"优化的 4-bit 量化格式
   假设权重 ~ N(0, σ²) → 在量化区间内更精细地表示"接近 0"的值
   → 比标准 INT4 精度更高

2. Double Quantization:
   对"量化常数的量化"
   如: 一个 64 元素 block 需要 1 个 scale (FP32=4B)
   Double Quantization: 把 scale 再量化到 FP8
   → 每个 block 省 3.5B (平均), 总共省 ~0.4 GB (65B 模型)

3. Paged Optimizer:
   用 CPU 内存做 unified memory paging
   → 处理训练时的 optimizer state 峰值显存

4. LoRA on all Linear layers:
   不仅仅是 Q/K/V, 所有 Linear 层都加 LoRA
   → 弥补量化带来的精度损失
```

### 1.3 QLoRA 配置示例

```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",        # NF4 量化
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算用 BF16
    bnb_4bit_use_double_quant=True,   # Double Quantization
)

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b-hf",
    quantization_config=bnb_config,
    device_map="auto",  # 自动分配到可用的 GPU
)
```

## 2. DoRA — 方向-幅度分解 (Liu et al., ICML 2024)

### 2.1 核心洞察

```
标准 LoRA: ΔW = B @ A

DoRA: 把 ΔW 分解为 "方向" 和 "幅度" 两个独立组件

  W_new = m × (W_pretrained + B@A) / ||W_pretrained + B@A||
          ↑                              ↑
       幅度 (标量)                    方向 (归一化)

  其中:
    m: 可学习的 magnitude (每列一个标量)
    ||·||: L2 norm (归一化方向)

为什么这样设计?
  全量微调时, 观察到:
    1. 方向的改变比幅度的改变小得多 (方向稳定)
    2. 幅度的改变与方向有负相关 (方向变化大时幅度变化小)

  DoRA 解耦了这两个变化维度 → 学习更稳定
```

### 2.2 效果

| | LoRA | DoRA |
|---|---|---|
| 学习模式 | ΔW 直接学 | 方向 + 幅度 解耦学 |
| 参数量 | r×(d_in+d_out) | r×(d_in+d_out) + d_out (多 d_out 个标量) |
| 收敛速度 | 基准 | **更快** |
| 最终效果 | 基准 | **+1-2%** on average |
| 额外开销 | — | ~1% (magnitude 参数很少) |

## 3. AdaLoRA — 自适应 Rank (Zhang et al., ICLR 2024)

### 3.1 问题: 固定 Rank 是次优的

```
标准 LoRA: 所有层用同一个 rank
  Layer 0 (浅层, 编码基础特征): rank=16 → 可能太多了
  Layer 31 (深层, 编码复杂语义): rank=16 → 可能不够

AdaLoRA: 根据每层的"重要性"动态分配 rank
```

### 3.2 工作方式

```python
# AdaLoRA 的核心逻辑
class AdaLoRA:
    def __init__(self, total_budget, init_rank):
        self.budget = total_budget  # 总 rank budget (如所有层 rank 之和)
        self.ranks = {layer: init_rank for layer in layers}

    def adapt_ranks(self):
        """根据奇异值的贡献重新分配 rank"""
        for layer in layers:
            # 对 ΔW 做 SVD
            U, S, V = svd(layer.delta_W)

            # 计算每个奇异值的重要性
            importance = S ** 2  # 能量 = 奇异值的平方

            # 当前层的重要性 = top-k 奇异值的能量占比
            layer_importance[layer] = sum(importance[:rank]) / sum(importance)

        # 重新分配: 重要的层给更多 rank
        sorted_layers = sort_by_importance_desc()
        for layer in sorted_layers:
            ranks[layer] = total_budget * layer_importance[layer]
```

## 4. LoRA+ / rsLoRA — 修正梯度不对称

### 4.1 问题

```
标准 LoRA 用 A=Kaiming, B=0 初始化
→ 早期训练中, B 的梯度远大于 A 的梯度
→ A 更新太慢 → 收敛慢

LoRA+: 给 B 设一个较小的学习率 (如 lr_B = lr_A / 2^4 = lr_A / 16)
→ A 和 B 以相同"速度"学习

rsLoRA: 修正缩放因子
  output = original + (alpha / sqrt(r)) × B@A@x  ← 注意是 sqrt(r), 不是 r
→ 消除 rank 增大时梯度不稳定的问题
```

## 5. 变体对比速查

| 变体 | 核心改进 | 额外开销 | 最适合 |
|------|---------|---------|--------|
| **LoRA** (base) | B@A 分解 | — | 通用起点 |
| **QLoRA** | NF4 量化 + Double Quant | 精度微降 | 消费级 GPU 微调大模型 |
| **DoRA** | 方向-幅度解耦 | ~1% | 追求更高精度 |
| **AdaLoRA** | 自适应 rank 分配 | SVD 计算 | 参数量受限时 |
| **LoRA+** | 不对称学习率 | 无 | 加速收敛 |
| **rsLoRA** | sqrt(r) 缩放 | 无 | 大 rank 训练稳定 |